# PopOut Game
This project aims to develop an AI capable of playing PopOut, a variant of Connect-4.

There should be three playing modes: AI vs AI, AI vs HUMAN and HUMAN vs HUMAN.

We implement:

A Monte Carlo Tree Search (MCTS) agent;
A Decision Tree (ID3) for learning from gameplay data;
We evaluate both approaches and analyze their performance.

## Constraints & Assumptions

We document below the constraints and modelling choices guiding our solution, as required by the brief.

**Algorithmic constraints**
- The ID3 decision tree is implemented entirely from scratch (no `scikit-learn`); third-party libraries are used only for I/O and visualization (`numpy`, `pandas`, `matplotlib`, `seaborn`).
- MCTS uses Upper Confidence Bound for Trees (UCT) as the branch-evaluation function, with the exploration constant `c` and the expansion count exposed as tunable hyperparameters.

**Game-rule modelling**
- Rule 1 — if a POP creates a four-in-a-row for both players, the player who popped wins. We resolve this in `Board.get_winner()` by checking the *last mover* first.
- Rule 2 — if the board is full, the player to move chooses between making a POP or declaring a draw. Implemented via `Board.is_full()` plus `Player.wants_draw()` (humans are prompted; bots default to *keep playing* so MCTS is not destabilised by a non-board-changing "declare draw" action).
- Rule 3 — threefold repetition of the same `(board, player-to-move)` state is declared a draw, tracked via `Board.state_history`.

**Dataset & evaluation**

- **PopOut dataset generation.** The supervised dataset is produced by *self-play* of DT-Wide (the tournament-winning MCTS variant) playing against itself. Each game contributes ~13 rows on average, one per move, with the columns being the 42 board cells before the move (values 0/1/2) and the move label (e.g. `DROP_3`, `POP_1`). Across 1,185 self-played games we collected 15,553 rows. Self-play has the advantage of producing only "sensible" moves (MCTS picks them), so the tree learns from a curated distribution rather than from random play.

- **80/20 train/test split with a fixed seed.** We hold out 20% of the rows for testing and train ID3 on the remaining 80%. Reporting only training accuracy would tell us nothing about *generalisation* — a tree deep enough to memorise the training set would score 100% on it while being useless on unseen states. The test set is the held-out portion that the tree has *never seen during fitting*, so its accuracy on it is our honest estimate of how well the tree would predict in real games. We fix the shuffle seed so that re-running the notebook produces the same split (and therefore the same numbers in the discussion below) instead of a different split every time.

- **Iris discretization (continuous → categorical).** ID3 in its textbook form splits on *categorical* attributes ("colour = red?"). The iris features, by contrast, are *continuous* (petal length 1.4 cm, 2.1 cm, …) — there is no finite list of values to enumerate. Our solution: for each feature, sort the observed values, then *every midpoint between two consecutive distinct values is a candidate threshold*. For each candidate `t` we compute the information gain of the binary split (`feature ≤ t` vs `feature > t`) and keep the threshold that maximises gain. This is the standard ID3-on-continuous-data trick and produces small, interpretable trees on iris without us having to pre-bin the data by hand.

- **PopOut accuracy = move-matching, not play quality.** This is the most important caveat to understand when reading our numbers. On the PopOut test set, ID3's "accuracy" is the fraction of rows where *the tree picks the same move MCTS picked*. That is **not** the same as the tree being a strong player. Example: in a given board state, both `DROP_3` and `DROP_4` may be excellent moves. MCTS happened to play `DROP_3` in our dataset; if ID3 outputs `DROP_4` we count that as wrong, even though it is a perfectly good move. So move-matching accuracy *under-estimates* play quality. A fairer benchmark would be a head-to-head match between an ID3-driven `BotPlayer` and `DT-Wide`, measured in win-rate. We revisit this caveat in the conclusion.

- **Random baseline of 1/14 ≈ 7%.** A PopOut move is one of 7 DROP columns or 7 POP columns = **14 possible labels** (ignoring legality for the moment). A model that guesses uniformly at random would score about 1/14 ≈ 7% accuracy. We draw this baseline as a dotted line on the accuracy-vs-depth chart so the reader can immediately see *how much better than random* the tree is at each depth — it gives the absolute accuracy numbers some scale. A depth-1 tree, for instance, sits just above 7%, which makes sense: a single binary split cannot disambiguate 14 different labels.


## Game Implementation

### BOARD

#### Game Representation

The game is represented using a 6x7 grid. Each cell can take values:
- 0 → empty
- 1 → player 1
- 2 → player 2

The game allows two types of moves:
- DROP: place a piece from the top
- POP: remove a piece from the bottom

The board is updated after every move.

The dataset was generated from recorded gameplay, where each board configuration was paired with the corresponding move taken by a player or agent.

In [ ]:
import numpy as np
from game.move import Move, MoveType

class Board:
    def __init__(self):
        # 6x7 board backed by NumPy
        self.current_player = 1
        self.grid = np.zeros((6, 7), dtype=np.uint8)
        
        estado_inicial = (self.grid.tobytes(), self.current_player)
        self.state_history = {estado_inicial: 1}

    def get_legal_moves(self) -> list[Move]:
        moves = []
        # 1. DROP: columns whose top row (0) is empty
        for c in range(7):
            if self.grid[0, c] == 0:
                moves.append(Move(move_type=MoveType.DROP, col=c))

        # 2. POP: columns where the bottom piece (row 5) belongs to the current player
        for c in range(7):
            if self.grid[5, c] == self.current_player:
                moves.append(Move(move_type=MoveType.POP, col=c))
        return moves

    def apply_move(self, move: Move) -> 'Board':
        new_board = Board.__new__(Board)
        new_board.grid = self.grid.copy()
        new_board.current_player = self.current_player
        
        # pass the history to the MCTS copies so they don't lose memory
        new_board.state_history = self.state_history.copy()

        if move.move_type == MoveType.DROP:
            for r in reversed(range(6)):
                if new_board.grid[r, move.col] == 0:
                    new_board.grid[r, move.col] = self.current_player
                    break

        elif move.move_type == MoveType.POP:
            # PopOut rule: remove the bottom piece and shift the column down
            new_board.grid[1:6, move.col] = new_board.grid[0:5, move.col]
            new_board.grid[0, move.col] = 0

        new_board.current_player = 3 - self.current_player
        
        state_key = (new_board.grid.tobytes(), new_board.current_player)
        new_board.state_history[state_key] = new_board.state_history.get(state_key, 0) + 1
        
        return new_board

    def is_full(self) -> bool:
        # No DROPs available means the board is full (top row has no empty cell).
        # POPs may still be legal — Rule 2 lets the player to move pop or declare draw.
        return not np.any(self.grid[0] == 0)

    def get_winner(self) -> int | None:
        # Player who just moved
        last_player = 3 - self.current_player

        def check(p):
            # Horizontal
            for r in range(6):
                for c in range(4):
                    if np.all(self.grid[r, c:c+4] == p): return True
            # Vertical
            for r in range(3):
                for c in range(7):
                    if np.all(self.grid[r:r+4, c] == p): return True
            # Diagonals
            for r in range(3):
                for c in range(4):
                    if all(self.grid[r+i, c+i] == p for i in range(4)): return True
                    if all(self.grid[r+3-i, c+i] == p for i in range(4)): return True
            return False

        # Rule 1: if a pop creates a four-in-a-row for both players, the mover wins
        if check(last_player): return last_player
        if check(self.current_player): return self.current_player

        # Stalemate fallback: no DROPs and no POPs are legal → automatic draw.
        # (Briefing Rule 2 — the "board full, player chooses pop or draw" rule —
        # is enforced in Game.run / Game.run_silent via is_full() + Player.wants_draw().)
        if not self.get_legal_moves(): return 0
        
        # Rule 3: draw by threefold repetition (infinite loops)
        estado_atual = (self.grid.tobytes(), self.current_player)
        if self.state_history.get(estado_atual, 0) >= 3:
            return 0  # 0 means draw

        return None

    def __str__(self):
        symbols = {0: ".", 1: "X", 2: "0"}
        header = " ".join(str(c + 1) for c in range(7))
        rows = "\n".join(" ".join(symbols[cell] for cell in row) for row in self.grid)
        return f"{header}\n{rows}"

### GAME

#### Game Logic

A player wins if they align 4 pieces:
- Horizontally
- Vertically
- Diagonally

Other rules:
- If both players connect 4 after a pop, the current player wins
- Draw if no moves available

In [ ]:
from game.board import Board
from game.player import HumanPlayer, BotPlayer

class Game:
    def __init__(self):
        self.board = Board()
        self.players = []

    def setup_game_mode(self):
        print("1. Human vs Human\n2. Human vs Bot\n3. Bot vs Bot")
        mode = input("Choose mode: ")
        if mode == "1":
            self.players = [HumanPlayer(), HumanPlayer()]
        elif mode == "2":
            self.players = [HumanPlayer(), BotPlayer()]
        else:
            self.players = [BotPlayer(), BotPlayer()]

    def run(self):
        self.setup_game_mode()
        curr_idx = 0

        while True:
            print("\n" + "=" * 30)
            print(f"Player {self.board.current_player}'s turn")
            print(self.board)
            # Rule 2: if the board is full, the player to move can declare a draw.
            if self.board.is_full() and self.players[curr_idx].wants_draw(self.board):
                print("\n" + "=" * 30)
                print(self.board)
                print("Draw declared by the player to move (Rule 2).")
                break
            move = self.players[curr_idx].get_move(self.board, False)
            self.board = self.board.apply_move(move)

            winner = self.board.get_winner()
            if winner is not None:
                print("\n" + "=" * 30)
                print(self.board)
                if winner == 0: print("Draw!")
                else: print(f"Player {winner} wins!")
                break
            curr_idx = 1 - curr_idx

    def run_silent(self, game_num: int = None, total_games: int = None):
        """Game loop without prints, for fast dataset generation / tournaments.

        When called with both game_num and total_games, prints a single
        progress line at the start so a long-running tournament shows life.
        Bot names come from each BotPlayer's MCTSConfig; humans show as 'Human'.
        """
        if game_num is not None and total_games is not None:
            def _name(p):
                # BotPlayer carries a config with a .name; HumanPlayer doesn't.
                return p.config.name if isinstance(p, BotPlayer) else "Human"
            p1, p2 = self.players[0], self.players[1]
            print(f"[{_name(p1)} vs {_name(p2)}] Game {game_num}/{total_games}")

        curr_player_idx = 0
        while True:
            curr_player = self.players[curr_player_idx]
            # Rule 2: same draw-declaration option in silent mode (bots default to keep playing).
            if self.board.is_full() and curr_player.wants_draw(self.board):
                return 0
            move = curr_player.get_move(self.board, False)
            self.board = self.board.apply_move(move)

            result = self.board.get_winner()
            if result is not None:
                break
            curr_player_idx = 1 - curr_player_idx
        return result

### MOVE

In [ ]:
from dataclasses import dataclass
from enum import Enum

# The two move types allowed in PopOut
MoveType = Enum('MoveType', [("DROP", "d"), ("POP", "p")])

@dataclass
class Move:
    # Lightweight container that carries a single move's information
    move_type: MoveType
    col: int

### PLAYER

In [ ]:
from mcts.mcts import mcts_search, MCTSConfig
from game.move import Move, MoveType

class Player:
    def get_move(self, board, is_game_drawable):
        pass

    def wants_draw(self, board) -> bool:
        # Rule 2: when the board is full, the player to move may declare a draw
        # instead of popping. Default policy: keep playing.
        return False

class BotPlayer(Player):
    def __init__(self, config: MCTSConfig | None = None):
        # Default config keeps existing call sites (BotPlayer()) working;
        # the tournament passes a specific MCTSConfig per variant.
        self.config = config if config is not None else MCTSConfig()

    def get_move(self, board, is_game_drawable) -> Move:
        return mcts_search(board, self.config)

class HumanPlayer(Player):
    def wants_draw(self, board) -> bool:
        print("\nThe board is full. You may POP or declare the game a draw.")
        while True:
            ans = input("Declare draw? (y/n): ").strip().lower()
            if ans in ("y", "yes"): return True
            if ans in ("n", "no"): return False
            print("Please answer 'y' or 'n'.")

    def get_move(self, board, is_game_drawable) -> Move:
        legal = board.get_legal_moves()
        legal_set = {(m.move_type, m.col) for m in legal}
        drops = sorted(m.col + 1 for m in legal if m.move_type == MoveType.DROP)
        pops = sorted(m.col + 1 for m in legal if m.move_type == MoveType.POP)
        print(f"Legal moves -> DROP cols: {drops}, POP cols: {pops}")
        while True:
            parts = input("Your move (e.g. 'd 3' or 'p 1'): ").strip().lower().split()
            if len(parts) != 2 or parts[0] not in ("d", "p") or not parts[1].isdigit():
                print("Invalid format. Use '<d|p> <col>', e.g. 'd 3'.")
                continue
            mt = MoveType.DROP if parts[0] == "d" else MoveType.POP
            col = int(parts[1]) - 1
            if (mt, col) not in legal_set:
                print("That move is not legal. Try again.")
                continue
            return Move(move_type=mt, col=col)

### Supported game modes

The brief requires the program to support three play modes, all wired up through `Game.setup_game_mode()`:

1. **Human vs Human** — two `HumanPlayer` instances; both moves come from CLI input.
2. **Human vs Computer** — one `HumanPlayer` and one `BotPlayer` (MCTS-driven).
3. **Computer vs Computer** — two `BotPlayer` instances, each configurable with its own `MCTSConfig`. This is the mode used by the tournament runner and by `main_datasets.py` to generate the PopOut dataset.

An interactive game can be launched from a terminal with `python main.py` (which calls `Game().run()` and prompts for the mode).


## Monte Carlo Tree Search

In [ ]:
import math
import random
import csv
import copy
import os
from dataclasses import dataclass
from game.board import Board


@dataclass
class MCTSConfig:
    # Bundle of knobs that defines an MCTS "variant" so the tournament
    # can spawn several bots that differ only by their config values.
    name: str = "Default"          # human-readable label e.g. "Baseline"
    iterations: int = 1000         # how many MCTS iterations per move
    c: float = 1.414               # UCT exploration constant
    expansion_count: int = 1       # how many children to expand per iteration


class MCTSNode:
    def __init__(self, board, parent=None, move=None):
        self.board = board
        self.parent = parent
        self.move = move
        self.children = []
        self.wins = 0
        self.visits = 0
        # Legal moves provided by Board
        self.untried_moves = board.get_legal_moves()

    def uct_value(self, c=1.414):
        if self.visits == 0:
            return float('inf')
        # UCT: exploitation + exploration
        return (self.wins / self.visits) + c * math.sqrt(math.log(self.parent.visits) / self.visits)


def _simulate_and_backprop(start_node, root_player):
    # Random playout from start_node.board, then walk back to the root
    # crediting wins/visits. Factored out so the iteration loop can call
    # it once per expanded child when expansion_count > 1.
    sim_board = copy.deepcopy(start_node.board)
    while sim_board.get_winner() is None:
        possible_moves = sim_board.get_legal_moves()
        if not possible_moves:
            break
        sim_board = sim_board.apply_move(random.choice(possible_moves))

    result = sim_board.get_winner()
    node = start_node
    while node:
        node.visits += 1
        # Win for the player who started the MCTS search
        if result == root_player:
            node.wins += 1
        elif result == 0:  # Draw
            node.wins += 0.5
        node = node.parent


def mcts_search(root_board, config: MCTSConfig):
    root_node = MCTSNode(root_board)

    for _ in range(config.iterations):
        node = root_node

        # 1. SELECTION
        while not node.untried_moves and node.children:
            node = max(node.children, key=lambda n: n.uct_value(config.c))

        # 2. EXPANSION (+ 3. SIMULATION + 4. BACKPROPAGATION per expanded child)
        if node.untried_moves:
            # Expand up to expansion_count children, or however many untried
            # moves remain. Each expanded child gets its own playout, so a
            # "wide" config (expansion_count > 1) does multiple rollouts per
            # iteration -- that means more total compute, not just a wider tree.
            n_expand = min(config.expansion_count, len(node.untried_moves))
            for _ in range(n_expand):
                move = node.untried_moves.pop()
                new_state = node.board.apply_move(move)
                child_node = MCTSNode(new_state, parent=node, move=move)
                node.children.append(child_node)
                _simulate_and_backprop(child_node, root_board.current_player)
        else:
            # Terminal selected node (no untried moves and no children either):
            # nothing to expand, so simulate from the node itself. Matches the
            # behaviour of the original single-expansion code.
            _simulate_and_backprop(node, root_board.current_player)

    # Pick the move with the most visits
    best_child = max(root_node.children, key=lambda n: n.visits)

    save_to_dataset(root_board, best_child.move)

    return best_child.move


_POPOUT_HEADER = ['game_id'] + [f'r{r}c{c}' for r in range(6) for c in range(7)] + ['move']
_POPOUT_PATH = 'popout_dataset.csv'
_current_game_id = 0

def set_game_id(gid: int):
    global _current_game_id
    _current_game_id = gid

def save_to_dataset(board, move):
    # Flatten the board (42 cells) and append the chosen move.
    # Writes the header row automatically if the file doesn't exist yet.
    data = board.grid.flatten().tolist()
    move_str = f"{move.move_type.name}_{move.col}"
    write_header = not os.path.exists(_POPOUT_PATH)
    with open(_POPOUT_PATH, 'a', newline='') as f:
        writer = csv.writer(f)
        if write_header:
            writer.writerow(_POPOUT_HEADER)
        writer.writerow([_current_game_id] + data + [move_str])

### Each iteration consists of:
1. Selection — traverse the tree using UCT
2. Expansion — add new child nodes
3. Simulation — play random games
4. Backpropagation — update node statistics


## MCTS Tournament

### Round-robin tournament between MCTS variants

Each variant is an MCTS with different configuration (different iterations / exploration constant /
expansion width). 

Every variant plays every other variant in both directions
(A as player 1 vs B as player 2, then B as player 1 vs A as player 2), so for
N variants we run N*(N-1) matchups, each of `games_per_matchup` games. 

Results are written to a CSV.

In [ ]:
import sys
import os
import csv
import time

# Make the repo's src/ importable when this file is run directly,
# the same trick main_datasets.py uses.
sys.path.append(os.path.join(os.path.dirname(__file__), '..'))

from mcts.mcts import MCTSConfig
from game.player import BotPlayer
from game.game import Game


# ── Tournament 1 variants ────────────────────────────────────────────────────
# Iterations were reduced from the original spec (500/2000/500/500) because at
# those values a full 50-game tournament took ~40h on this machine.
# Pilot (2026-05-07): avg 6.7s/game → 100 games/matchup ≈ 2h 14min total.
VARIANTS = [
    MCTSConfig(name="Baseline",     iterations=50,  c=1.414, expansion_count=1),
    MCTSConfig(name="DeepThinker",  iterations=200, c=1.414, expansion_count=1),
    MCTSConfig(name="Explorer",     iterations=50,  c=2.5,   expansion_count=1),
    MCTSConfig(name="WideExpander", iterations=50,  c=1.414, expansion_count=3),
]

SMOKE_VARIANTS = [
    MCTSConfig(name="Baseline",     iterations=10, c=1.414, expansion_count=1),
    MCTSConfig(name="DeepThinker",  iterations=20, c=1.414, expansion_count=1),
    MCTSConfig(name="Explorer",     iterations=10, c=2.5,   expansion_count=1),
    MCTSConfig(name="WideExpander", iterations=10, c=1.414, expansion_count=3),
]

# ── Tournament 2 variants ────────────────────────────────────────────────────
# DeepThinker won tournament 1. These four sub-variants hold iterations=200
# fixed and vary c and expansion_count to find the optimal configuration.
# DT-Frugal is the control (exact tournament 1 winner).
# Per-game timing is the same as T1 → 100 games/matchup ≈ 2h 14min.
VARIANTS_T2 = [
    MCTSConfig(name="DT-Frugal",  iterations=200, c=1.414, expansion_count=1),
    MCTSConfig(name="DT-Exploit", iterations=200, c=0.7,   expansion_count=1),
    MCTSConfig(name="DT-Explore", iterations=200, c=2.0,   expansion_count=1),
    MCTSConfig(name="DT-Wide",    iterations=200, c=1.414, expansion_count=2),
]

SMOKE_VARIANTS_T2 = [
    MCTSConfig(name="DT-Frugal",  iterations=10, c=1.414, expansion_count=1),
    MCTSConfig(name="DT-Exploit", iterations=10, c=0.7,   expansion_count=1),
    MCTSConfig(name="DT-Explore", iterations=10, c=2.0,   expansion_count=1),
    MCTSConfig(name="DT-Wide",    iterations=10, c=1.414, expansion_count=2),
]

# ── Output paths ─────────────────────────────────────────────────────────────
_ROOT = os.path.join(os.path.dirname(__file__), '..', '..')
RESULTS_PATH    = os.path.join(_ROOT, 'first_tournament_results.csv')
RESULTS_PATH_T2 = os.path.join(_ROOT, 'second_tournament_results.csv')


def _load_existing(results_path, matchups):
    """Read existing CSV tallies so this run appends rather than restarts.

    Returns (tally, games_so_far) where games_so_far is the number of games
    already played per matchup. If the file doesn't exist, or a matchup pair
    is missing from it, that pair starts at zero.
    """
    tally = {(b1.name, b2.name): [0, 0, 0] for b1, b2 in matchups}
    games_so_far = 0
    if not os.path.exists(results_path):
        return tally, 0
    with open(results_path, newline='') as f:
        for row in csv.DictReader(f):
            key = (row['bot1_name'], row['bot2_name'])
            if key in tally:
                tally[key] = [int(row['bot1_wins']), int(row['bot2_wins']), int(row['draws'])]
                games_so_far = int(row['total_games'])
    return tally, games_so_far


def _save_csv(tally, games_played_per_matchup, results_path):
    # Overwrite the file with current standings -- always exactly one row per
    # matchup. Called after every round so a crash loses at most one round.
    with open(results_path, 'w', newline='') as f:
        writer = csv.writer(f)
        writer.writerow(['bot1_name', 'bot2_name', 'bot1_wins', 'bot2_wins', 'draws', 'total_games'])
        for (n1, n2), (w, l, d) in tally.items():
            writer.writerow([n1, n2, w, l, d, games_played_per_matchup])


def run_tournament(variants, games_per_matchup=50, results_path=RESULTS_PATH):
    # Build the ordered matchup list once.
    # A vs B and B vs A are separate because Pop Out isn't symmetric.
    matchups = [
        (bot1, bot2)
        for bot1 in variants
        for bot2 in variants
        if bot1 is not bot2
    ]
    total_games_all = len(matchups) * games_per_matchup
    games_done = 0
    tournament_start = time.perf_counter()

    # Load any existing results so this run accumulates on top of prior runs.
    tally, games_so_far = _load_existing(results_path, matchups)
    if games_so_far:
        print(f"Resuming from existing results ({games_so_far} games/matchup already played).")

    # Interleaved loop: play one game of every matchup (a "round"), then
    # repeat. After each round the CSV is overwritten with cumulative standings.
    # If the run crashes, every matchup has the same number of games --
    # the snapshot is always representative.
    for round_i in range(games_per_matchup):
        for bot1, bot2 in matchups:
            game = Game()
            game.players = [BotPlayer(bot1), BotPlayer(bot2)]

            t0 = time.perf_counter()
            result = game.run_silent(game_num=round_i + 1, total_games=games_per_matchup)
            t_game = time.perf_counter() - t0

            key = (bot1.name, bot2.name)
            if result == 1:
                tally[key][0] += 1
            elif result == 2:
                tally[key][1] += 1
            else:
                tally[key][2] += 1

            games_done += 1
            t_total = time.perf_counter() - tournament_start
            avg = t_total / games_done
            eta = avg * (total_games_all - games_done)
            print(f"  → done in {t_game:.1f}s | avg {avg:.1f}s/game | "
                  f"{games_done}/{total_games_all} games | ETA: ~{eta:.0f}s")

        # Persist after every round and print a summary block.
        # total_games = prior runs + rounds completed this run.
        _save_csv(tally, games_so_far + round_i + 1, results_path)
        print("#" * 20)
        print(f"Round {round_i + 1}/{games_per_matchup} complete — current standings:")
        for (n1, n2), (w, l, d) in tally.items():
            print(f"  {n1} vs {n2}  W:{w} L:{l} D:{d}")
        print("#" * 20)

    t_total = time.perf_counter() - tournament_start
    avg = t_total / total_games_all
    projection = avg * 1200  # projection for 100 games/matchup (useful during pilots)
    proj_h = int(projection // 3600)
    proj_m = int((projection % 3600) // 60)
    print(f"\nTournament complete. Results saved to {results_path}")
    print(f"Total time: {t_total:.0f}s | Average per game: {avg:.1f}s")
    print(f"→ For 100 games/matchup (1200 total): estimate {projection:.0f}s (~{proj_h}h {proj_m}m)")


if __name__ == "__main__":
    # Tournament 2: DeepThinker sub-variants, 100 games/matchup, ~2h 14min.
    run_tournament(VARIANTS_T2, games_per_matchup=100, results_path=RESULTS_PATH_T2)

### Tournament Visualization 
Reads experiments_results.csv and produces:
  - heatmap.png  : win-rate matrix (row bot vs column bot)
  - barchart.png : overall win rate per variant

In [ ]:
import sys
import os

# Same sys.path trick as tournament.py so this runs as a standalone script.
sys.path.append(os.path.join(os.path.dirname(__file__), '..'))

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# All plots land here so they stay out of the repo root.
GRAPHS_DIR = os.path.join(os.path.dirname(__file__), 'graphs')


def load_results(filepath: str) -> pd.DataFrame:
    """Load experiments_results.csv into a DataFrame."""
    return pd.read_csv(filepath)


def plot_win_rate_heatmap(df: pd.DataFrame, output_path: str = None) -> None:
    """
    Generate a win rate heatmap matrix where cell (A, B) shows
    the win rate of bot A against bot B. Save to output_path.
    """
    if output_path is None:
        output_path = os.path.join(GRAPHS_DIR, 'heatmap.png')

    # Compute win rate for each (bot1, bot2) pair.
    # Cells on the diagonal don't exist (no self-play) and stay NaN.
    data = df.copy()
    data['win_rate'] = data['bot1_wins'] / data['total_games']
    pivot = data.pivot(index='bot1_name', columns='bot2_name', values='win_rate')

    _, ax = plt.subplots(figsize=(8, 6))
    sns.heatmap(
        pivot,
        annot=True,       # print the number inside each cell
        fmt='.2f',        # two decimal places
        cmap='RdYlGn',    # red = bot loses, green = bot wins
        vmin=0, vmax=1,
        linewidths=0.5,
        ax=ax,
    )
    ax.set_title('Win rate: row bot (player 1) vs column bot (player 2)')
    ax.set_xlabel('Opponent (bot2)')
    ax.set_ylabel('Challenger (bot1)')
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close()
    print(f'Heatmap saved to {output_path}')


def plot_win_rate_barchart(df: pd.DataFrame, output_path: str = None) -> None:
    """
    Generate a bar chart showing the overall win rate of each variant
    across all matchups combined. Save to output_path.
    """
    if output_path is None:
        output_path = os.path.join(GRAPHS_DIR, 'barchart.png')

    # For each variant (as player 1), total wins / total games played.
    agg = (
        df.groupby('bot1_name')
        .agg(total_wins=('bot1_wins', 'sum'), total_played=('total_games', 'sum'))
        .reset_index()
    )
    agg['win_rate'] = agg['total_wins'] / agg['total_played']
    agg = agg.sort_values('win_rate', ascending=False)

    _, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(agg['bot1_name'], agg['win_rate'], color='steelblue', edgecolor='white')

    # Label each bar with the exact win rate.
    for bar, rate in zip(bars, agg['win_rate']):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.02,
            f'{rate:.2f}',
            ha='center', va='bottom', fontsize=10,
        )

    ax.set_ylim(0, 1.15)
    ax.set_ylabel('Win rate (as player 1)')
    ax.set_title('Overall win rate per MCTS variant')
    # Dotted line at 50% so it's easy to see who beats random chance.
    ax.axhline(0.5, color='gray', linestyle='--', linewidth=1, label='50%')
    ax.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close()
    print(f'Bar chart saved to {output_path}')


def generate_all(results_path: str = "first_tournament_results.csv") -> None:
    """Load results and generate all plots."""
    df = load_results(results_path)
    plot_win_rate_heatmap(df)
    plot_win_rate_barchart(df)


if __name__ == "__main__":
    generate_all()

### Analysis

- DeepThinker performs better due to more simulations -> DeepThinker had 4x more iterations than the others.
- Explorer explores more but may be less consistent
- WideExpander increases branching but also computation cost

The tournament results show that DeepThinker (more iterations) consistently beat Explorer (greater 'c' constant).
Which allows us to conclude that, on PopOut, depth of search (lookahead) is more effective than random exploration

While the Explorer spends time simulating suboptimal branches, DeepThinker uses its computational budget to statistically validate the strongest lines of play.

## ID3 - Decision Tree
The ID3 algorithm constructs a decision tree by recursively selecting the feature that maximizes information gain, which is computed using entropy.

We used it to build a supervised learning model that predicts moves in the PopOut game. 

The goal was to create a data-driven agent capable of selecting actions based on previously observed game states.

In [ ]:
import math
import csv
import random
from graphviz import Digraph
from collections import Counter


class Node:
    def __init__(self):
        self.feature = None         # feature name to split on (None at leaves)
        self.threshold = None       # float threshold for continuous splits; None for discrete
        self.children = {}          # discrete: {value: Node}; continuous: {'<=': Node, '>': Node}
        self.label = None           # class label at a leaf node (None at internal nodes)
        self.majority_label = None  # fallback if predict hits an unseen feature value


def entropy(labels) -> float:
   
    if not labels:
        return 0.0
    counts = Counter(labels)
    total = len(labels)
    return -sum((c / total) * math.log2(c / total) for c in counts.values())


def information_gain(data: list[dict], feature: str, label_col: str,
                     threshold: float | None = None) -> float:
   
    labels = [row[label_col] for row in data]
    parent_h = entropy(labels)
    n = len(data)

    if threshold is None:
        partitions = {}
        for row in data:
            partitions.setdefault(row[feature], []).append(row[label_col])
        weighted = sum((len(p) / n) * entropy(p) for p in partitions.values())
    else:
        left  = [row[label_col] for row in data if row[feature] <= threshold]
        right = [row[label_col] for row in data if row[feature] >  threshold]
        weighted = (len(left) / n) * entropy(left) + (len(right) / n) * entropy(right)

    return parent_h - weighted


def best_split(data: list[dict], features: list[str], label_col: str,
               numerical_features: set = frozenset()) -> tuple[str, float | None]:

    best_gain = -1.0
    best_feature = None
    best_threshold = None

    for feature in features:
        if feature in numerical_features:
            values = sorted(set(row[feature] for row in data))
            thresholds = [(values[i] + values[i + 1]) / 2 for i in range(len(values) - 1)]
            for t in thresholds:
                gain = information_gain(data, feature, label_col, threshold=t)
                if gain > best_gain:
                    best_gain, best_feature, best_threshold = gain, feature, t
        else:
            gain = information_gain(data, feature, label_col)
            if gain > best_gain:
                best_gain, best_feature, best_threshold = gain, feature, None

    return best_feature, best_threshold


def _majority(labels: list) -> str:
    """Return the most common label; alphabetical order breaks ties."""
    counts = Counter(labels)
    return min(counts, key=lambda l: (-counts[l], l))


def build_tree(data: list[dict], features: list[str], label_col: str,
               numerical_features: set = frozenset(),
               max_depth: int | None = None, depth: int = 0) -> Node:
    """Recursively build an ID3 decision tree.

    Stopping conditions (produce a leaf):
    - All examples share the same label (pure node).
    - No features left to split on.
    - max_depth reached.
    """
    node = Node()
    labels = [row[label_col] for row in data]
    node.majority_label = _majority(labels)

    # Pure node
    if len(set(labels)) == 1:
        node.label = labels[0]
        return node

    # No features left or depth limit reached
    if not features or (max_depth is not None and depth >= max_depth):
        node.label = node.majority_label
        return node

    feature, threshold = best_split(data, features, label_col, numerical_features)

    if feature is None:
        node.label = node.majority_label
        return node

    node.feature = feature
    node.threshold = threshold

    if threshold is None:
        # Discrete split — remove this feature from future splits on this branch
        remaining = [f for f in features if f != feature]
        partitions = {}
        for row in data:
            partitions.setdefault(row[feature], []).append(row)

        for val, subset in partitions.items():
            node.children[val] = build_tree(
                subset, remaining, label_col, numerical_features, max_depth, depth + 1
            )
    else:
        # Continuous split — feature stays available for deeper splits
        left  = [row for row in data if row[feature] <= threshold]
        right = [row for row in data if row[feature] >  threshold]

        for key, subset in [('<=', left), ('>', right)]:
            if subset:
                node.children[key] = build_tree(
                    subset, features, label_col, numerical_features, max_depth, depth + 1
                )
            else:
                leaf = Node()
                leaf.label = node.majority_label
                node.children[key] = leaf

    return node


def predict(node: Node, example: dict) -> str:
    """Walk the tree and return the predicted class label for `example`."""
    if node.label is not None:
        return node.label

    val = example[node.feature]

    if node.threshold is None:
        child = node.children.get(val)
        if child is None:
            return node.majority_label  # unseen value at test time
        return predict(child, example)
    else:
        key = '<=' if val <= node.threshold else '>'
        return predict(node.children[key], example)


def print_tree(node: Node, indent: int = 0, branch_label: str = "",
               max_print_depth: int | None = None) -> None:
    """Print an indented text representation of the tree.

    max_print_depth limits how many levels to print (useful for large PopOut trees).
    """
    if max_print_depth is not None and indent > max_print_depth:
        print("  " * indent + "...")
        return

    prefix = "  " * indent
    tag = f"[{branch_label}] " if branch_label else ""

    if node.label is not None:
        print(f"{prefix}{tag}→ {node.label}")
        return

    if node.threshold is not None:
        print(f"{prefix}{tag}{node.feature} <= {node.threshold:.3f}?")
        print_tree(node.children.get('<='), indent + 1, "<=", max_print_depth)
        print_tree(node.children.get('>'),  indent + 1, ">",  max_print_depth)
    else:
        print(f"{prefix}{tag}{node.feature}?")
        for val in sorted(node.children.keys(), key=str):
            print_tree(node.children[val], indent + 1, str(val), max_print_depth)


def evaluate(node: Node, test_data: list[dict], label_col: str) -> float:
    """Return accuracy (0.0–1.0) of the tree on `test_data`."""
    if not test_data:
        return 0.0
    correct = sum(1 for row in test_data if predict(node, row) == row[label_col])
    return correct / len(test_data)


# ── Helpers ───────────────────────────────────────────────────────────────────

def tree_stats(node: Node) -> dict:
    """Return depth, total nodes, internal nodes, and leaf count for the tree."""
    if node is None:
        return {'depth': 0, 'nodes': 0, 'leaves': 0, 'internal': 0}
    if node.label is not None:
        return {'depth': 0, 'nodes': 1, 'leaves': 1, 'internal': 0}
    child_stats = [tree_stats(child) for child in node.children.values()]
    return {
        'depth':    1 + max(s['depth']    for s in child_stats),
        'nodes':    1 + sum(s['nodes']    for s in child_stats),
        'leaves':       sum(s['leaves']   for s in child_stats),
        'internal': 1 + sum(s['internal'] for s in child_stats),
    }


def train_test_split(data: list[dict], test_ratio: float = 0.2,
                     seed: int = 42) -> tuple[list[dict], list[dict]]:
    """Shuffle data and split into (train, test) sets."""
    data = list(data)
    random.seed(seed)
    random.shuffle(data)
    split = int(len(data) * (1 - test_ratio))
    return data[:split], data[split:]


def load_csv(filepath: str) -> list[dict]:
    """Load a CSV with a header row into a list of dicts."""
    with open(filepath, newline='') as f:
        return list(csv.DictReader(f))


def save_results(results: list[dict], filepath: str) -> None:
    """Save a list of result dicts to a CSV file."""
    if not results:
        return
    with open(filepath, 'w', newline='') as f:
        writer = csv.DictWriter(f, fieldnames=results[0].keys())
        writer.writeheader()
        writer.writerows(results)

def export_to_vscode(node, filename="arvore_iris", max_depth=None):
    """Gera um ficheiro Markdown (.md) que o VS Code desenha automaticamente."""
    lines = ["# Visualização da Árvore de Decisão", "", "```mermaid", "graph TD"]

    def build_mermaid(node, node_id="Node0", depth=0):
        if node is None or (max_depth is not None and depth > max_depth):
            return
        
        # Se for folha (Resultado final) - Fica Verde
        if node.label is not None:
            label = str(node.label).replace("->", "").strip()
            # (["texto"]) faz uma caixa com cantos arredondados
            lines.append(f'    {node_id}(["{label}"])')
            lines.append(f'    style {node_id} fill:#a7f0a7,stroke:#333,stroke-width:2px')
            return

        # Nó de decisão (Pergunta) - Fica Azul
        condicao = "<=" if node.threshold else ""
        label_principal = f"{node.feature} {condicao} {node.threshold:.2f}" if node.threshold else node.feature
        
        # ["texto"] faz uma caixa quadrada
        lines.append(f'    {node_id}["{label_principal}"]')
        lines.append(f'    style {node_id} fill:#a7d8f0,stroke:#333,stroke-width:2px')
        
        # Desenhar as setas
        for i, (branch, child) in enumerate(node.children.items()):
            child_id = f"{node_id}_{i}"
            txt_ramo = "Sim" if str(branch) == "<=" else "Não" if str(branch) == ">" else str(branch)
            lines.append(f'    {node_id} -- "{txt_ramo}" --> {child_id}')
            build_mermaid(child, child_id, depth + 1)

    build_mermaid(node)
    lines.append("```")
    
    with open(f"{filename}.md", "w", encoding="utf-8") as f:
        f.write("\n".join(lines))
    print(f"[INFO] Abre o ficheiro '{filename}.md' no VS Code para veres a árvore!")

if __name__ == "__main__":
    import os, sys, time
    
    # Garante que o Python encontra os ficheiros
    base_path = os.path.dirname(__file__)
    sys.path.append(os.path.join(base_path, '..', '..'))

    print("=" * 40)
    print("FASE 1: Demonstração ID3 — Iris Dataset")
    print("=" * 40)

    iris_path = os.path.join(base_path, '..', '..', 'datasets', 'iris-dataset.csv')
    if os.path.exists(iris_path):
        data = load_csv(iris_path)
        for row in data:
            if 'ID' in row: del row['ID']
        
        numerical = {'sepallength', 'sepalwidth', 'petallength', 'petalwidth'}
        features = [f for f in data[0] if f != 'class']
        for row in data:
            for f in numerical: row[f] = float(row[f])

        train, test = train_test_split(data, test_ratio=0.2)
        tree = build_tree(train, features, 'class', numerical_features=numerical)
        
        # GERAR ÁRVORE
        export_to_vscode(tree, filename="arvore_iris_grafica")

        print("\nÁrvore Iris (Texto):")
        print_tree(tree)
        print(f"\nIris Accuracy: {evaluate(tree, test, 'class'):.2%}")

    print("\n" + "=" * 40)
    print("FASE 2: Processamento PopOut")
    print("=" * 40)

    popout_path = 'popout_dataset.csv'
    if not os.path.exists(popout_path):
        popout_path = os.path.join(base_path, '..', '..', 'popout_dataset.csv')

    if os.path.exists(popout_path):
        p_data = load_csv(popout_path)
        p_features = [f for f in p_data[0] if f not in ('game_id', 'move')]
        for row in p_data:
            for f in p_features: row[f] = int(row[f])

        p_train, p_test = train_test_split(p_data, test_ratio=0.2)
        DEPTHS = [1, 3, 5, 7, 10, 15, 20, 30, None]
        results = []

        for depth in DEPTHS:
            p_tree = build_tree(p_train, p_features, 'move', max_depth=depth)
            
            # GERAR ÁRVORE (SÓ PROFUNDIDADE 3)
            if depth == 3:
                export_to_vscode(p_tree, filename="arvore_popout_grafica_depth3", max_depth=3)

            train_acc = evaluate(p_tree, p_train, 'move')
            test_acc = evaluate(p_tree, p_test, 'move')
            results.append({'max_depth': depth, 'train_acc': train_acc, 'test_acc': test_acc})
            
            label = str(depth) if depth is not None else 'unlimited'
            print(f"  > Depth {label:>9}: Test={test_acc:.2%}")

        save_results(results, 'id3_popout_results.csv')
        print("\n" + "=" * 40)
        print("[SUCESSO] Abre os ficheiros .md no VS Code para veres os gráficos!")
        print("=" * 40)

### ID3 Visualization results on the PopOut dataset.

Reads id3_popout_results.csv and produces:
    popout_accuracy_vs_depth.png

In [ ]:
import sys
import os

sys.path.append(os.path.join(os.path.dirname(__file__), '..'))

import pandas as pd
import matplotlib.pyplot as plt

GRAPHS_DIR = os.path.join(os.path.dirname(__file__), 'graphs')


def load_results(filepath: str) -> pd.DataFrame:
    return pd.read_csv(filepath)


def plot_accuracy_vs_depth(df: pd.DataFrame, output_path: str = None) -> None:
    """Bar chart of test accuracy per max_depth value."""
    if output_path is None:
        os.makedirs(GRAPHS_DIR, exist_ok=True)
        output_path = os.path.join(GRAPHS_DIR, 'popout_accuracy_vs_depth.png')

    df = df.copy()
    df['max_depth'] = df['max_depth'].fillna('unlimited').astype(str)

    _, ax = plt.subplots(figsize=(8, 5))
    bars = ax.bar(df['max_depth'], df['test_accuracy'], color='steelblue', edgecolor='white')

    for bar, acc in zip(bars, df['test_accuracy']):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.02,
            f'{acc:.2f}',
            ha='center', va='bottom', fontsize=10,
        )

    ax.set_ylim(0, 1.15)
    ax.set_xlabel('max_depth')
    ax.set_ylabel('Test accuracy')
    ax.set_title('ID3 on PopOut — test accuracy vs tree depth')
    # Dotted line at random baseline (1 / number of possible moves)
    ax.axhline(1 / 14, color='gray', linestyle='--', linewidth=1, label='random baseline (~7%)')
    ax.legend()
    plt.tight_layout()
    plt.savefig(output_path, dpi=150)
    plt.close()
    print(f'Plot saved to {output_path}')


def generate_all(results_path: str = 'id3_popout_results.csv') -> None:
    df = load_results(results_path)
    plot_accuracy_vs_depth(df)


if __name__ == "__main__":
    generate_all()

## Conclusion: ID3 vs MCTS

### What we learned about MCTS
Across two tournaments (8 variants, hundreds of head-to-head games per matchup), the dominant signal was that **search depth beats exploration breadth** on PopOut with a fixed compute budget. In Tournament 1, *DeepThinker* (200 iterations) clearly outperformed the 50-iteration variants regardless of whether the latter widened exploration (`c=2.5`) or expanded more children per node. In Tournament 2 we then probed sub-variants of the 200-iteration setup and *DT-Wide* (`c=1.414`, `expansion_count=2`) edged out the others — modest extra breadth helps once you already have enough depth, but a high `c` or many child expansions on shallow searches just dilute the simulations. We therefore picked DT-Wide as the canonical PopOut bot and used it to generate the supervised dataset.

### What we learned about ID3
On iris (4 features, 150 rows), ID3 produces a small, interpretable tree (depth 4, 11 nodes, 100% train / 90% test accuracy) — a sanity check that the algorithm, the entropy/information-gain math, and the continuous-feature discretization are all wired correctly. On PopOut (42 features, ~15k rows), the depth sweep `[1, 3, 5, 7, 10, 15, 20, 30, None]` produces the bar chart above. Two things are visible in that curve:

- **Shallow trees underfit badly** — at `max_depth=1` accuracy hovers just above the 1/14 ≈ 7% random baseline, because a single feature cannot disambiguate 14 different move labels from a 42-cell board.
- **Test accuracy plateaus quickly** — beyond `max_depth≈10` the tree stops gaining test-set accuracy even as the train-set accuracy keeps climbing. That gap is the classic overfitting signature: deeper trees memorise individual `(state, move)` rows without learning more general structure.

### Caveat: move-matching ≠ play quality
The ID3 accuracy reported here measures **agreement with MCTS labels**, not actual win rate against an opponent. A move that disagrees with MCTS may still be perfectly playable, and conversely matching MCTS on this particular dataset doesn't prove the tree generalises to game states the bot hasn't seen. A more honest evaluation would be a head-to-head match (ID3-driven `BotPlayer` vs `DT-Wide`) measured in win-rate — see the brief discussion of this in `notebook_notes.md`.

### Tradeoffs and recommendation
MCTS and ID3 are answering different questions:

| | MCTS (DT-Wide) | ID3 |
|---|---|---|
| Decision basis | Forward simulation of consequences | Pattern recognition over the current board |
| Inference cost | Seconds per move (200 iterations of rollout) | O(depth) — microseconds |
| Interpretability | Opaque — depends on the random rollouts | Fully inspectable tree (Mermaid renders above) |
| Generalisation | Re-derives the right move every turn | Limited to patterns seen during training |

MCTS is the stronger player; ID3 is the faster, cheaper, and explainable approximation of one. For high-level competitive play we would deploy MCTS (DT-Wide); for a resource-constrained or latency-sensitive context — embedded, real-time, or for a human-readable rationale — ID3 is the better fit, especially with `max_depth` tuned to the elbow of the accuracy curve above to avoid overfitting.
